# PI IV: classificação com LLM no Google Colab (GPU)

Este notebook roda o `classificar_llm.py` com o Ollama na GPU gratuita do Colab.

**Antes de começar:**
1. No Google Drive, crie a pasta `PI_IV` e coloque nela:
   - `classificar_llm.py`
   - `vagas_enriquecidas.csv` (gerado pelo `enriquecimento_regras.py`)
2. Aqui no Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.
3. Rode as células **em ordem**.

O cache (`cache_*.jsonl`) é salvo direto no Drive. Se o Colab desconectar, rode de novo as células 1 a 4 e a etapa que parou. Ela continua de onde estava.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "SEM GPU: troque o ambiente de execução para T4"

## 2. Montar o Google Drive e entrar na pasta do projeto

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

PASTA = "/content/drive/MyDrive/PI_IV"   # ajuste se a pasta tiver outro nome
os.chdir(PASTA)

for arq in ["classificar_llm.py", "vagas_enriquecidas.csv"]:
    print(("OK      " if os.path.exists(arq) else "FALTANDO") + "  " + arq)

## 3. Instalar e iniciar o Ollama

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("Ollama rodando")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama não subiu. Veja o log:")
    print(open("/content/ollama.log").read()[-2000:])

## 4. Baixar o modelo

Com GPU, o `qwen2.5:7b` é rápido e mais preciso que o 3b. Confira se o `MODELO` no `classificar_llm.py` é o mesmo baixado aqui.

In [ ]:
!ollama pull qwen2.5:7b
!grep -n "^MODELO" classificar_llm.py

## 5. Validação (rode primeiro)

Anote a **acurácia** e a matriz de confusão para a metodologia do relatório.
Confira também se o `ollama ps` mostra `100% GPU`.

In [ ]:
!python -u classificar_llm.py validar senioridade 200
!ollama ps

## 6. Classificar a senioridade (~5,9 mil vagas)

Com GPU, deve levar ~1–2 s por vaga. **Deixe a aba do Colab aberta** para evitar desconexão por inatividade.

In [ ]:
!python -u classificar_llm.py rodar senioridade

## 7. Classificar o contexto de IA (~1,4 mil vagas)

In [ ]:
!python -u classificar_llm.py rodar ia_contexto

## 8. Juntar os resultados

Gera o `vagas_enriquecidas_llm.csv` na pasta do Drive, pronto para o Power BI.

In [ ]:
!python -u classificar_llm.py juntar
!ls -lh *.csv cache_*.jsonl

---
### Se algo der errado

| Sintoma | O que fazer |
|---|---|
| `Connection refused` | o Ollama caiu: rode de novo a célula 3 |
| Colab desconectou | rode de novo as células 2, 3 e 4 e depois a etapa que parou; o cache continua no Drive |
| `ollama ps` mostra `CPU` | o ambiente não está com GPU: refaça o passo 2 da introdução e reinicie |
| Muito lento (> 5 s por vaga) | confira a GPU com `!nvidia-smi` |